# Module 1 — BlackEvent Index EDA

Module 1 실행 결과를 시각적으로 확인하는 노트북.
- 코스피 시계열 + BlackEvent 구간 하이라이트
- BlackEvent 연도별 분포 바 차트
- max_cumulative_drop 히스토그램
- gap_days 탐색 결과 비교

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

plt.rcParams["font.family"] = "Malgun Gothic"  # Windows 한글 폰트
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

DATA_RAW = Path("../data/raw")
DATA_PROC = Path("../data/processed")

In [ ]:
# 데이터 로드
kospi = pd.read_csv(DATA_RAW / "kospi_daily.csv", parse_dates=["Date"])
events = pd.read_csv(DATA_PROC / "black_events.csv", parse_dates=["first_shock_date", "last_shock_date"])
control = pd.read_csv(DATA_PROC / "control_dates.csv", parse_dates=["date"])

print(f"코스피 데이터: {len(kospi)}일 ({kospi['Date'].min().date()} ~ {kospi['Date'].max().date()})")
print(f"BlackEvent: {len(events)}건")
print(f"대조군: {len(control)}건")
events.head(10)

## 1. 코스피 시계열 + BlackEvent 구간 하이라이트

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))

# 코스피 종가 시계열
ax.plot(kospi["Date"], kospi["Close"], color="#2c3e50", linewidth=0.7, label="KOSPI 종가")

# BlackEvent 구간 하이라이트
for _, ev in events.iterrows():
    ax.axvspan(ev["first_shock_date"], ev["last_shock_date"],
               alpha=0.3, color="red", linewidth=0)

ax.set_title("KOSPI 종가 시계열 — BlackEvent 구간 하이라이트 (빨강)", fontsize=14)
ax.set_xlabel("날짜")
ax.set_ylabel("종가")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 2. BlackEvent 연도별 분포

In [ ]:
events["year"] = events["first_shock_date"].dt.year
year_counts = events["year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(year_counts.index, year_counts.values, color="#e74c3c", edgecolor="#c0392b")
ax.set_title("BlackEvent 연도별 발생 건수", fontsize=14)
ax.set_xlabel("연도")
ax.set_ylabel("건수")
ax.set_xticks(year_counts.index)

for x, y in zip(year_counts.index, year_counts.values):
    ax.text(x, y + 0.2, str(y), ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## 3. 누적 최대 낙폭 (max_cumulative_drop) 히스토그램

In [ ]:
drops = events["max_cumulative_drop"].dropna()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(drops, bins=20, color="#e67e22", edgecolor="#d35400", alpha=0.85)
ax.axvline(drops.median(), color="red", linestyle="--", label=f"중앙값: {drops.median():.1f}%")
ax.axvline(drops.mean(), color="blue", linestyle="--", label=f"평균: {drops.mean():.1f}%")
ax.set_title("BlackEvent 누적 최대 낙폭 분포", fontsize=14)
ax.set_xlabel("max_cumulative_drop (%)")
ax.set_ylabel("빈도")
ax.legend()
plt.tight_layout()
plt.show()

print(f"낙폭 통계:")
print(drops.describe())

## 4. gap_days 탐색 결과 비교

In [ ]:
gap_path = DATA_PROC / "gap_days_search.csv"
if gap_path.exists():
    gap_df = pd.read_csv(gap_path)
    display(gap_df)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].bar(gap_df["gap_days"], gap_df["n_events"], color="#3498db")
    axes[0].set_title("gap_days별 BlackEvent 수")
    axes[0].set_xlabel("gap_days")
    axes[0].set_ylabel("이벤트 수")

    axes[1].bar(gap_df["gap_days"], gap_df["avg_duration"], color="#2ecc71")
    axes[1].set_title("gap_days별 평균 지속 기간")
    axes[1].set_xlabel("gap_days")
    axes[1].set_ylabel("일")

    axes[2].bar(gap_df["gap_days"], gap_df["avg_cumulative_drop"], color="#e74c3c")
    axes[2].set_title("gap_days별 평균 낙폭")
    axes[2].set_xlabel("gap_days")
    axes[2].set_ylabel("%")

    plt.tight_layout()
    plt.show()
else:
    print("gap_days_search.csv가 없습니다. Module 1을 먼저 실행하세요.")

## 5. 주요 BlackEvent 목록

In [ ]:
# 낙폭이 큰 순서로 Top 15
top = events.nsmallest(15, "max_cumulative_drop")[
    ["event_id", "first_shock_date", "duration_days", "shock_count", "max_cumulative_drop"]
]
top["first_shock_date"] = top["first_shock_date"].dt.date
display(top)